In [0]:
df_processed = spark.read.format("delta") \
    .load("/Volumes/workspace/default/api/processed/")

display(df_processed)


# ✅ 1. Avg Temperature & Humidity per City

from pyspark.sql.functions import avg

df_gold = df_processed.groupBy("city").agg(
    avg("temperature").alias("avg_temperature"),
    avg("humidity").alias("avg_humidity")
)

display(df_gold)


# ✅ 2. Latest Weather per City
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("city").orderBy(col("event_time").desc())

df_latest = df_processed.withColumn("rank", row_number().over(window)) \
                        .filter("rank = 1") \
                        .drop("rank")

display(df_latest)

In [0]:
    df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/api/curated/weather_agg")

dbutils.fs.rm("/Volumes/workspace/default/api/curated/weather_latest", True)

df_latest.write \
    .mode("overwrite") \
    .format("delta") \
    .save("/Volumes/workspace/default/api/curated/weather_latest")

df_gold.show()
df_latest.show()

In [0]:
df_gold.write \
    .mode("overwrite") \
    .saveAsTable("weather_gold_agg")

df_latest.write \
    .mode("overwrite") \
    .saveAsTable("weather_gold_latest")

In [0]:
%sql
DESCRIBE HISTORY weather_gold_agg;

In [0]:
%sql
SELECT * FROM weather_gold_agg VERSION AS OF 0;

In [0]:
%sql
SELECT * FROM weather_gold_agg;